# 04 · Zipping, Archives & Compression

**Module 11 – Files & OS** · Estimated time: 30 min

> Updates 📗 *Unzipping and Zipping Files* (`zipfile`, `shutil.make_archive`) with `tarfile`, `gzip`, and the 🆕 Python 3.14 `compression` package (Zstandard).

### 🎯 What you will learn
- Creating and reading `.zip` files with `zipfile`
- Archiving whole folders with `shutil.make_archive` / `unpack_archive`
- `.tar.gz` archives with `tarfile`
- Compressing single files/streams with `gzip`, `bz2`, `lzma`
- 🆕 `compression.zstd` (3.14): fast, modern Zstandard compression

## 1. Create files to compress (📗)

In [1]:
with open("fileone.txt", "w", encoding="utf-8") as f:
    f.write("ONE FILE " * 200)
with open("filetwo.txt", "w", encoding="utf-8") as f:
    f.write("TWO FILE " * 200)

## 2. Zipping files with `zipfile`

In [2]:
import zipfile

with zipfile.ZipFile("comp_file.zip", "w", compression=zipfile.ZIP_DEFLATED) as comp_file:
    comp_file.write("fileone.txt")
    comp_file.write("filetwo.txt")
    comp_file.writestr("notes/readme.txt", "written directly from a string")

import os
print(os.path.getsize("fileone.txt") + os.path.getsize("filetwo.txt"), "bytes →",
      os.path.getsize("comp_file.zip"), "bytes zipped")

3600 bytes → 408 bytes zipped


> 💡 **Compression method:** without a `compression=` argument (or a per-file `compress_type=`, as the bootcamp passes to each `write()`), `ZipFile` uses `ZIP_STORED`, which means **no compression at all**. Setting `compression=zipfile.ZIP_DEFLATED` once on the `ZipFile` applies to every file (alternatives: `ZIP_LZMA`, `ZIP_BZIP2`; 🆕 `ZIP_ZSTANDARD` in 3.14).
>
> ⚠️ **Updated from the bootcamp:** it opens/closes the archive manually (`comp_file.close()`) and hard-codes Windows paths like `'C:\\Users\\Marcial\\...'`. Use `with` and relative paths or `pathlib` instead.

## 3. Inspecting and extracting

In [3]:
with zipfile.ZipFile("comp_file.zip") as zf:
    for info in zf.infolist():
        print(f"{info.filename:<22} {info.file_size:>6} → {info.compress_size:>5} bytes")
    print(zf.read("notes/readme.txt").decode())
    zf.extract("fileone.txt", "extracted_one")        # a single file
    zf.extractall("extracted_content")                 # everything

print(sorted(os.listdir("extracted_content")))

fileone.txt              1800 →    25 bytes
filetwo.txt              1800 →    25 bytes
notes/readme.txt           30 →    32 bytes
written directly from a string
['fileone.txt', 'filetwo.txt', 'notes']


> ⚠️ **Security:** never `extractall()` an archive from an untrusted source without checking it. Malicious archives can contain paths like `../../etc/passwd` ("zip slip"). `zipfile` sanitises names, but `tarfile` needs a filter (below).

## 4. Whole folders with `shutil` (📗)

In [4]:
import shutil

os.makedirs("project/src", exist_ok=True)
with open("project/src/app.py", "w", encoding="utf-8") as f:
    f.write("print('hello')\n")
with open("project/README.md", "w", encoding="utf-8") as f:
    f.write("# Project\n")

# make_archive(output name WITHOUT extension, format, folder to archive)
archive = shutil.make_archive("project_backup", "zip", "project")
print(archive)
shutil.unpack_archive("project_backup.zip", "project_restored", "zip")
print([str(p) for p in sorted(__import__("pathlib").Path("project_restored").rglob("*"))])
print(shutil.get_archive_formats())

/private/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/tmp2r7s9jl3/project_backup.zip
['project_restored/README.md', 'project_restored/src', 'project_restored/src/app.py']
[('bztar', "bzip2'ed tar-file"), ('gztar', "gzip'ed tar-file"), ('tar', 'uncompressed tar file'), ('xztar', "xz'ed tar-file"), ('zip', 'ZIP file'), ('zstdtar', "zstd'ed tar-file")]


## 5. `.tar.gz` with `tarfile` (common on Linux/macOS)

In [5]:
import tarfile

with tarfile.open("project.tar.gz", "w:gz") as tar:
    tar.add("project", arcname="project")

with tarfile.open("project.tar.gz", "r:gz") as tar:
    print(tar.getnames())
    tar.extractall("from_tar", filter="data")     # ✅ filter="data" blocks dangerous paths (3.12+, default in 3.14)

['project', 'project/README.md', 'project/src', 'project/src/app.py']


## 6. Compressing single files & bytes: `gzip`, `bz2`, `lzma`

In [6]:
import gzip, bz2, lzma

data = ("Python is fun! " * 1000).encode()
for name, module in [("gzip", gzip), ("bz2", bz2), ("lzma", lzma)]:
    packed = module.compress(data)
    assert module.decompress(packed) == data
    print(f"{name:<5} {len(data)} → {len(packed)} bytes")

with gzip.open("log.txt.gz", "wt", encoding="utf-8") as f:     # "t" = text mode
    f.write("line 1\nline 2\n")
with gzip.open("log.txt.gz", "rt", encoding="utf-8") as f:
    print(f.read())

gzip  15000 → 82 bytes
bz2   15000 → 79 bytes
lzma  15000 → 132 bytes
line 1
line 2



## 7. 🆕 `compression.zstd`: Zstandard (Python 3.14, PEP 784)

**Zstandard** (by Meta) compresses about as well as gzip or better, and is **much faster**. It's used by Linux package managers, databases and web servers. Python 3.14 adds it to the standard library under a new `compression` package (which also re-exports `gzip`, `bz2`, `lzma`, `zlib` as `compression.gzip`, etc.).

In [7]:
from compression import zstd
import time

big = ("2026-10-20 INFO user=asha action=login ok\n" * 50_000).encode()
for name, comp, decomp in [("gzip", gzip.compress, gzip.decompress),
                           ("zstd", zstd.compress, zstd.decompress)]:
    t = time.perf_counter()
    packed = comp(big)
    elapsed = time.perf_counter() - t
    assert decomp(packed) == big
    print(f"{name}: {len(big):,} → {len(packed):,} bytes in {elapsed * 1000:.1f} ms")

gzip: 2,100,000 → 6,190 bytes in 4.8 ms
zstd: 2,100,000 → 256 bytes in 0.5 ms


In [8]:
with zstd.open("data.txt.zst", "wt", encoding="utf-8") as f:
    f.write("compressed with Zstandard\n")
with zstd.open("data.txt.zst", "rt", encoding="utf-8") as f:
    print(f.read())

with zipfile.ZipFile("modern.zip", "w", compression=zipfile.ZIP_ZSTANDARD) as zf:   # 3.14+
    zf.write("fileone.txt")
print(zipfile.ZipFile("modern.zip").infolist()[0].compress_type == zipfile.ZIP_ZSTANDARD)

compressed with Zstandard

True


## 📝 Summary
- `zipfile.ZipFile(..., "w", compression=ZIP_DEFLATED)` to create; `.extractall()` to unpack.
- `shutil.make_archive` / `unpack_archive` for whole folders.
- `tarfile` for `.tar.gz`; use `filter="data"` when extracting.
- `gzip`/`bz2`/`lzma` for single streams; 🆕 `compression.zstd` (3.14) for fast modern compression.

## 🏋️ Practice (📗 Advanced Modules Puzzle, full version in `exercises/`)
1. Zip every `.txt` file in the current folder into `texts.zip` (compressed).
2. Print the compression ratio of each file inside `texts.zip`.
3. Back up a folder to `backup_YYYY-MM-DD.zip` using `shutil.make_archive` and today's date.

<details><summary>Show solutions</summary>

```python
import zipfile, shutil
from pathlib import Path
from datetime import date

with zipfile.ZipFile("texts.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in Path(".").glob("*.txt"):
        zf.write(p)

with zipfile.ZipFile("texts.zip") as zf:
    for i in zf.infolist():
        print(i.filename, f"{i.compress_size / max(i.file_size, 1):.0%}")

shutil.make_archive(f"backup_{date.today()}", "zip", "project")
```
</details>